In [ ]:
import os
# Suppress TF C++ warnings (0=all, 1=no info, 2=no info/warnings)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import keras
import keras_tuner as kt
import tensorflow as tf
import matplotlib.pyplot as plt

from learning_to_rank.data import build_dataset
from learning_to_rank.tuning.hypermodels import TransformerRankerHyperModel
from learning_to_rank.callbacks import TransformerWarmupCallback

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

In [ ]:
CONFIG = {
    "train_path": "data/tfrecords/train.tfrecord",
    "val_path": "data/tfrecords/val.tfrecord",
    "project_name": "transformer_notebook_tuning",
    "max_epochs": 50,
    "warmup_epochs": 10,
    "batch_size": 32,
    "num_features": 136
}

In [ ]:
train_ds = build_dataset(CONFIG["train_path"], batch_size=CONFIG["batch_size"], shuffle=True)
val_ds = build_dataset(CONFIG["val_path"], batch_size=CONFIG["batch_size"], shuffle=False)

# Peek at one batch
for x_batch, y_batch in train_ds.take(1):
    print(f"Input batch shape: {x_batch.shape}")  # [Batch, List, Features]
    print(f"Label batch shape: {y_batch.shape}")  # [Batch, List]

In [ ]:
hypermodel = TransformerRankerHyperModel(num_features=CONFIG["num_features"])

tuner = kt.Hyperband(
    hypermodel,
    objective=kt.Objective("val_ndcg", direction="max"),
    max_epochs=CONFIG["max_epochs"],
    factor=3,
    directory="models/tuning",
    project_name=CONFIG["project_name"],
    executions_per_trial=2 
)

callbacks = [
    TransformerWarmupCallback(warmup_epochs=CONFIG["warmup_epochs"]),
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)
]

In [ ]:
print(f"Starting Transformer Optimization...")
tuner.search(
    train_ds,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1 # Changed to 1 for notebook progress bars
)

In [ ]:
# Display summary of results
tuner.results_summary()

# Retrieve best model
best_model = tuner.get_best_models(num_models=1)[0]
best_model.save("models/best_transformer_notebook.keras")

print("[SUCCESS] Best Transformer model saved.")